# 2. Data & Preprocessing

## Loading the dataset

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("akyshnik/lyrics")

print("Path to dataset files:", path)

Path to dataset files: /home/alberto/.cache/kagglehub/datasets/akyshnik/lyrics/versions/1


In [2]:
import os

print(os.listdir(path))

['kaggle_clean.csv']


In [3]:
import pandas as pd

file_path = os.path.join(path, "kaggle_clean.csv")
df = pd.read_csv(file_path)

## 2.1 Dataset Selection

### Checking the values

In [7]:
print(df["genre"].unique())
df.info(),
df.sample(10)

['Pop' 'Hip-Hop' 'Not Available' 'Rock' 'Metal' 'Other' 'Country' 'Jazz'
 'Electronic' 'Folk' 'R&B' 'Indie']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266557 entries, 0 to 266556
Data columns (total 8 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   Unnamed: 0  266557 non-null  int64 
 1   index       266557 non-null  int64 
 2   song        266556 non-null  object
 3   year        266557 non-null  int64 
 4   artist      266557 non-null  object
 5   genre       266557 non-null  object
 6   lyrics      266535 non-null  object
 7   wnum        266557 non-null  int64 
dtypes: int64(4), object(4)
memory usage: 16.3+ MB


,Unnamed: 0,index,song,year,artist,genre,lyrics,wnum
232392,232392,315364,scrap-metal,2007,bitch-animal,Indie,how do i feel?\nits funny you should ask\ni st...,620
75353,75353,104855,desperados-waiting-for-the-train,2007,david-allan-coe,Country,i'd sing the red river valley\nand he'd sit in...,290
105987,105987,144934,wild-chase,2006,edenbridge,Metal,the dewdrop of dawn\nare glaring in morningly ...,124
240085,240085,326333,top-of-the-world,2007,charley-pride,Country,how's the view from the top of the world i won...,98
87102,87102,120470,sex-you-up,2013,chris-brown,Hip-Hop,what the hell babe\ndamn i ain't never felt th...,428
188179,188179,256396,without-you,2006,debbie-gibson,Pop,verse 1:\ncouldn't go anywhere\nwithout feelin...,215
169104,169104,230515,street-life,2007,b-b-king,Rock,i still hang around\nneither lost nor found\nh...,353
170252,170252,232124,can-t-hold-us-down,2014,axwell-i-ingrosso,Other,can't hold us\ncan't hold us\ncan't hold us\nc...,195
127217,127217,173703,a-thank-you,2007,cool-hand-luke,Rock,you've gone and done it again\ntaken everythin...,209
173590,173590,236716,the-saddest-day,2007,converge,Metal,and we won't be breathing in that same sun aga...,132


In [ ]:
print(len(df))
df = df.dropna(subset=["lyrics"])
print(len(df))

### Drop columns with useless information

In [6]:
df = df.drop(columns=["Unnamed: 0", "index"])

### Modify useless values

We modify every value from the lyrics that are formed by one word to be "instrumental"

In [7]:
df.loc[df["wnum"] == 1, "lyrics"].value_counts()

lyrics
instrumental     3645
instru             51
instumental        10
intro              10
intrumental         9
                 ... 
orchestra           1
finntroll           1
pow!                1
instermentual       1
bonus               1
Name: count, Length: 73, dtype: int64

In [8]:
df.loc[df["wnum"] == 1, "lyrics"] = "instrumental"
df.loc[df["wnum"] == 1, "lyrics"].value_counts()

lyrics
instrumental    3842
Name: count, dtype: int64

### Eliminate Non Available values

We do this because there is enough values to being able to eliminate every row that has incomplete information.

In [9]:
df_clean = df[~df["genre"].isin(["Not Available"])]
df_clean = df_clean[~df_clean["year"].isin([112, 702, 67])]

In [10]:
print(len(df), len(df_clean))

266557 242610


### Check impact of instrumental songs

In [11]:
print("Percentage of instrumental songs for each genre:\n")

inst = df_clean["genre"][df_clean["lyrics"] == "instrumental"].value_counts()
no_inst = df_clean["genre"][df_clean["lyrics"] != "instrumental"].value_counts()
genres = inst.keys()

for i in range(len(inst)):
    print(genres[i]+": ", round((inst.iloc[i]/no_inst.iloc[i])*100, 2), "\b%")

Percentage of instrumental songs for each genre:

Rock:  1.31%
Metal:  1.9%
Electronic:  1.4%
Pop:  0.88%
Jazz:  1.01%
Folk:  1.73%
Hip-Hop:  1.12%
Country:  1.23%
Indie:  1.33%
R&B:  0.32%
Other:  0.28%


### Check number of values of each genre

In [12]:
genre_values = df_clean["genre"].value_counts()
df_size = len(df_clean)
biggest_value = 0
for genre in genres:
    if len(genre) > biggest_value:
        biggest_value = len(genre)
for i in range(len(genres)):
    rounded_number = round((genre_values.iloc[i]/df_size)*100, 2)
    print(genres[i]+":"+" "*(biggest_value-len(genres[i]) + 5-len(str(rounded_number))), 
          rounded_number, "\b%\t", genre_values.iloc[i])

Rock:       45.02%	 109235
Metal:      16.68%	 40465
Electronic: 10.24%	 24845
Pop:         9.79%	 23759
Jazz:        5.93%	 14387
Folk:        3.29%	 7971
Hip-Hop:     3.28%	 7966
Country:     2.14%	 5189
Indie:        1.4%	 3401
R&B:          1.3%	 3149
Other:       0.92%	 2243


### Dataset Description

- Source: https://www.kaggle.com/datasets/akyshnik/lyrics
- Domain: Prediction of music genre from the lyrics.
- Label distribution:
| Genre | Percentage | Samples |
| --- | --- | --- |
| Rock | 54.02% | 109235 |
| Metal | 16.68% | 40465 |
| Electronic | 10.24% | 24845 |
| Pop | 9.79% | 23759 |
| Jazz | 5.93% | 14387 |
| Folk | 3.29% | 7971 |
| Hip-Hop | 3.28% | 7966 |
| Country | 2.14% | 5189 |
| Indie | 1.4% | 3401 |
| R&B | 1.3% | 3149 |
| Other | 0.92% | 2243 |
- Potential biases: The dataset can be biased towards Rock music, since almost half of the music from this dataset is Rock. But, even with this statement, the most represented genres (Rock, Metal, Electronic, Pop) have subgenres inside them (blues rock, goth metal, afro music, pop-punk). So it makes sense for these genres to have the most amount of samples.
- Ethical considerations: All the information of the dataset is public. Some songs contain offensive language, but this shouldn't be taken as harmful content, since offensive language also helps defining a way of expressing, thus helping on the genre prediction.

## 2.2 Data Split

### Data Split justification

For the data split, we will give 70% to the train dataset, 15% to the development dataset, and 15% to the test dataset.
The reason of this division is to train the dataset with the most amount of data, so the model can learn patterns correctly. For this reason, the development and test datasets should be big enough to work properly, but as small as possible to use the most amount of data on the training dataset.

The randomness is controlled via the random_state parameter. By applying a value on this parameter, we ensure that every time we repeat the executions, the dataset will split in the same way.

We also use stratification to keep the label proportions evenly in each dataset, since it wouldn't be ideal to have one of the datasets almost full of Rock songs, for example.

In [13]:
from sklearn.model_selection import train_test_split

df_train, df_temp = train_test_split(df_clean, test_size=0.3, stratify=df_clean["genre"], random_state=42)
df_dev, df_test = train_test_split(df_temp, test_size=0.5, stratify=df_temp["genre"], random_state=42)

In [14]:
print(len(df_clean), len(df_train), len(df_dev), len(df_test))

242610 169827 36391 36392


## 2.3 Preprocessing strategy

In [15]:
df_train.info()
df_train.sample(10)

<class 'pandas.core.frame.DataFrame'>
Index: 169827 entries, 265908 to 257161
Data columns (total 6 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   song    169826 non-null  object
 1   year    169827 non-null  int64 
 2   artist  169827 non-null  object
 3   genre   169827 non-null  object
 4   lyrics  169827 non-null  object
 5   wnum    169827 non-null  int64 
dtypes: int64(2), object(4)
memory usage: 9.1+ MB


,song,year,artist,genre,lyrics,wnum
97913,amarte-asi,2008,elvis-crespo,Pop,ven aquã­ ya me decidã­ abrir mi alma\nmi alma...,144
242849,the-feast,2006,church,Rock,there were many guests at the table\nmisfortun...,107
128496,ice-cream-van,2008,glasvegas,Rock,there's a storm on the horizon\nand for that i...,103
206544,get-you-in,2006,better-than-ezra,Rock,i sit and watch your flowers\nwilting in the k...,278
23640,we-rock,2008,demi-lovato,Pop,'cause we rock\nwe rock we rock on\nwe rock we...,290
52163,pendemic,2006,fat-joe,Hip-Hop,yeah uh\ni don't give a fuck fuck you\nfuck yo...,452
222628,touchdown,2007,do-or-die,Metal,chorus 2x\nwhen i touchdown uh uh oh oh mah ni...,445
260132,printemps,2009,coeur-de-pirate,Pop,et en 2003 dans un show en ã©tã©\nsous mes cou...,248
153227,celebrate,2006,fields-of-the-nephilim,Rock,when the moment's right\nonly moments rise\nfo...,533
103749,asi-soy-yo,2011,bustamante,Pop,asã­ soy yo\nes que en cada paso yo te siento ...,227
